In [3]:
"""
ab2.py - Adams-Bashforth 2nd Order Method (AB2)

Adams-Bashforth 2nd order method for solving ODEs.
Uses RK2 for the first step to get initial values.
"""

import numpy as np

def ab2(f_ode, xRange, yInitial, numSteps):
    """
    [x, y] = ab2(f_ode, xRange, yInitial, numSteps)
    
    Uses Adams-Bashforth second-order method to solve a system
    of first-order ODEs dy/dx = f_ode(x,y).
    
    Input:
    f_ode = evaluates right hand side.
    xRange = [x1, x2] where the solution is sought on x1 <= x <= x2
    yInitial = initial values for y at x1
    numSteps = number of equally-sized steps to take from x1 to x2
    
    Output:
    x = vector of values of x
    y = matrix whose k-th row is the approximate solution at x(k)
    """
    
    x = np.zeros(numSteps + 1)
    y = np.zeros((numSteps + 1, np.size(yInitial)))
    dx = (xRange[1] - xRange[0]) / numSteps
    
    for k in range(0, numSteps + 1):
        if k == 0:
            x[k] = xRange[0]
            y[k, :] = yInitial
        elif k == 1:
            # First step: Use RK2 (Euler halfstep) to get second point
            fValue = f_ode(x[k-1], y[k-1, :])
            xhalf = x[k-1] + 0.5 * dx
            yhalf = y[k-1, :] + 0.5 * dx * fValue
            fValuehalf = f_ode(xhalf, yhalf)
            x[k] = x[k-1] + dx
            y[k, :] = y[k-1, :] + dx * fValuehalf
            # Store fValue for next step (will be fValueold)
        else:
            # Adams-Bashforth 2nd order formula
            # y_{k+1} = y_k + h * (3*f(x_k, y_k) - f(x_{k-1}, y_{k-1})) / 2
            fValueold = fValue  # f at previous step (k-1)
            fValue = f_ode(x[k-1], y[k-1, :])  # f at current step (k)
            x[k] = x[k-1] + dx
            y[k, :] = y[k-1, :] + dx * (3.0 * fValue - fValueold) / 2.0
    
    if y.shape[1] == 1:
        y = y.flatten()
    
    return x, y

In [4]:
"""
exercise7.py - Exercise 7: Adams-Bashforth 2nd Order Method (AB2)
ODE: dy/dx = -y - 3x, y(0) = 1, x ∈ [0, 2]

Compares AB2 method with different step sizes and estimates order of accuracy.
"""

import numpy as np
from ab2 import ab2
from rk2 import rk2

# ============================================
# ODE AND EXACT SOLUTION
# ============================================
def expm_ode(x, y):
    """dy/dx = -y - 3x"""
    return -y - 3.0 * x

def exact_solution(x):
    """y(x) = -2e^(-x) - 3x + 3"""
    return -2.0 * np.exp(-x) - 3.0 * x + 3.0


# ============================================
# MAIN CODE
# ============================================

print("="*80)
print("EXERCISE 7: ADAMS-BASHFORTH 2ND ORDER METHOD (AB2)")
print("ODE: dy/dx = -y - 3x, y(0) = 1")
print("Interval: x ∈ [0, 2]")
print("="*80)

# Parameters
xRange = np.array([0.0, 2.0])
yInit = 1.0

# Step sizes
numSteps_list = [10, 20, 40, 80, 160, 320]
h_list = [2.0/n for n in numSteps_list]

exact_at_2 = exact_solution(2.0)

# ============================================
# TABLE 1: AB2 Results
# ============================================
print("\n" + "="*80)
print("TABLE 1: AB2 RESULTS")
print("="*80)
print(f"{'k':<3} {'numSteps':<10} {'Step size h':<12} {'Y[-1,0]':<18} {'Error E[k]':<18}")
print("-"*80)

errors = []
y_finals = []

for k, numSteps in enumerate(numSteps_list):
    h = h_list[k]
    x, y = ab2(expm_ode, xRange, yInit, numSteps)
    
    # Get final y value
    if y.ndim == 1:
        y_final = y[-1]
    else:
        y_final = y[-1, 0]
    
    y_finals.append(y_final)
    
    # Calculate error
    error = abs(y_final - exact_at_2)
    errors.append(error)
    
    print(f"{k:<3} {numSteps:<10} {h:<12.6f} {y_final:<18.10f} {error:<18.10e}")

print("-"*80)

# First line should match: k=0, y[-1,0] = -3.28013993, error = 9.4694e-03
print("\n✓ First line verification:")
print(f"  Expected: y(2) = -3.28013993, error = 9.4694e-03")
print(f"  Got:      y(2) = {y_finals[0]:.10f}, error = {errors[0]:.10e}")

# ============================================
# TABLE 2: Error Ratios
# ============================================
print("\n" + "="*80)
print("TABLE 2: AB2 ERROR RATIOS")
print("="*80)
print(f"{'k':<3} {'numSteps':<10} {'Error E[k]':<18} {'E[k]/E[k+1]':<15}")
print("-"*80)

ratios = []

for k in range(len(numSteps_list)):
    if k == 0:
        ratio_str = "---"
    else:
        ratio = errors[k-1] / errors[k]
        ratios.append(ratio)
        ratio_str = f"{ratio:.4f}"
    
    print(f"{k:<3} {numSteps_list[k]:<10} {errors[k]:<18.10e} {ratio_str:<15}")

print("-"*80)

# ============================================
# Estimate order of accuracy
# ============================================
print("\n" + "="*80)
print("ESTIMATE ORDER OF ACCURACY")
print("="*80)

avg_ratio = np.mean(ratios)
print(f"\nError ratios (E[k]/E[k+1]):")
for i, r in enumerate(ratios, 1):
    print(f"  E[{i-1}]/E[{i}] = {r:.4f}")

print(f"\nAverage ratio: {avg_ratio:.4f}")
print(f"Theoretical ratio for 2nd order method: 4.00 (since 2² = 4)")
print(f"Estimated order p = log2({avg_ratio:.4f}) = {np.log2(avg_ratio):.4f}")

if abs(np.log2(avg_ratio) - 2.0) < 0.2:
    print("\n✓ Conclusion: AB2 is 2nd order accurate (p = 2)")
else:
    print(f"\nConclusion: AB2 is approximately {np.log2(avg_ratio):.2f} order accurate")

# ============================================
# COMPARE WITH RK2 (from Exercise 2)
# ============================================
print("\n" + "="*80)
print("COMPARISON: AB2 vs RK2")
print("="*80)

# RK2 errors from Exercise 2
RK2_ERRORS = [4.2255e-03, 1.0451e-03, 2.6061e-04, 6.5121e-05, 1.6286e-05, 4.0713e-06]

print(f"\n{'k':<3} {'numSteps':<10} {'AB2 Error':<18} {'RK2 Error':<18}")
print("-"*50)

for k in range(len(numSteps_list)):
    print(f"{k:<3} {numSteps_list[k]:<10} {errors[k]:<18.10e} {RK2_ERRORS[k]:<18.10e}")

print("-"*50)

# ============================================
# OBSERVATIONS
# ============================================
print("\n" + "="*80)
print("OBSERVATIONS")
print("="*80)
print("""
1. AB2 is a 2nd order method (p = 2), same as RK2.
   
2. AB2 uses derivative values from previous steps (multistep method),
   while RK2 is a single-step method.
   
3. AB2 requires only 1 function evaluation per step after the first,
   while RK2 requires 2 evaluations per step.
   
4. AB2 is more efficient (computationally cheaper) than RK2 for the same
   number of steps, but has similar accuracy (both 2nd order).
   
5. The first step of AB2 uses RK2 to get started, which has O(h³) error,
   matching AB2's local truncation error.
""")

print("\n" + "="*80)
print("Exercise 7 completed!")
print("="*80)

<class 'ModuleNotFoundError'>: No module named 'ab2'